# Model D — Model C+ Handcrafted Features (HCF) 

```
front, back (300×300×3)
   ↓ EfficientNet-B3 (weight 공유)
feature map (1536, 10, 10)
   ↓ GLAM (weight 공유)
   ↓ GAP → flatten → (1536)
   ↓ Concat (3072)                            ← image feature
                                                
HCF (N차원)                                    ← extract_hcf_v2.ipynb 산출물
   ↓ HCF Encoder: BN → Linear(N→128) → GELU → Dropout
   ↓             → Linear(128→64) → GELU → Dropout
                                                  → (64)
   ↓ Concat (3072 + 64 = 3136)
   ↓ MLP (3136 → 1024 → 256 → 16 → 5)
CrossEntropyLoss → condition 1~5
```

## Model C와의 paired comparison
이 노트북은 **Model D와 단 하나의 변수만 다름**: HCF 사용 여부.

| 동일 | 다름 |
|---|---|
| Backbone (EfficientNet-B3), GLAM 구조, augmentation, optimizer 종류, lr, schedule, batch, seed, split | 1) HCF Encoder가 추가됨 (HCFEncoder, ~28K params)<br>2) Head 입력 차원 3072 → 3136 (HCF embed 64 concat)<br>3) Optimizer param group 3 → 4 (HCF 추가) |

Stage 1 (3 epoch) freeze 동작도 동일하지만, **HCF Encoder는 항상 학습**됩니다 (backbone과 별개).

## HCF 두 버전
| RUN_NAME | HCF_VERSION | HCF 차원 | 의미 |
|---|---|---|---|
| run_E_meta_seed_42_01 | `meta` | 148 | 배포 가능 (객관 정보만) |
| run_E_full_seed_42_01 | `full` | 183 | 상한선 (Model F 결과상 +10%p 누수) |

## 실행 방법
```python
# CFG의 HCF_VERSION을 바꿔서 두 번 Run All
HCF_VERSION = 'meta'  # 1회차
HCF_VERSION = 'full'  # 2회차
```
또는 환경변수로 자동 전환 (`os.environ['HCF_VERSION']`).


## 1. Setup

In [ ]:
# 필요 시 한 번만 실행
#!pip install torch torchvision timm coral-pytorch scikit-learn pandas opencv-python tqdm matplotlib seaborn

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTHONHASHSEED", os.environ.get("SEED", "42"))

import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms.functional as TF
import torchvision.transforms as T
import timm

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Stop here to avoid accidental CPU training. "
        "Check NVIDIA driver/CUDA state before running this notebook."
    )

device = torch.device("cuda:0")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Config

`WORKSPACE_ROOT`, `PRETRAINED_WEIGHTS` 경로만 본인 환경에 맞게 수정.


In [ ]:
class CFG:
    WORKSPACE_ROOT = Path('/workspace')
    MODEL_E_ROOT = WORKSPACE_ROOT / 'mint' / 'model_e'                  # 'model_d' → 'model_e'
    PREPROCESSED_DIR = WORKSPACE_ROOT / 'preprocessed_ssim09_blur10_300'
    MANIFESTS_DIR = PREPROCESSED_DIR / 'manifests'
    HCF_DIR       = WORKSPACE_ROOT / 'hcf'                     # NEW: extract_hcf_v2 산출물
    CHECKPOINT_DIR = MODEL_E_ROOT / 'checkpoints'

    IMG_SIZE      = 300
    NUM_CLASSES   = 5

    # ─── HCF 설정 (Model D 대비 추가) ───
    # 'meta' = 148차원 (배포 가능 정보만)
    # 'full' = 183차원 (+ damage 라벨, 상한선)
    HCF_VERSION    = os.environ.get('HCF_VERSION', 'meta').lower()
    assert HCF_VERSION in {'meta', 'full'}, f'HCF_VERSION must be meta or full, got {HCF_VERSION}'
    HCF_EMBED_DIM  = 64
    HCF_HIDDEN_DIM = 128
    HCF_DROPOUT    = 0.2

    MODEL_NAME    = f'E_multiview_efficientnet_b3_glam_hcf_{HCF_VERSION}'
    BACKBONE_NAME = 'efficientnet_b3'
    PRETRAINED    = True
    # D 팀 방식과 일치: hf_hub_download로 명시적 다운로드 후 safetensors load
    from huggingface_hub import hf_hub_download as _hf_hub_download
    PRETRAINED_WEIGHTS = Path(
        _hf_hub_download(
            repo_id="timm/efficientnet_b3.ra2_in1k",
            filename="model.safetensors"
        )
    )
    FEATURE_DIM   = 1536
    DROPOUT       = 0.3

    # GLAM 하이퍼파라미터 (Model D와 완전히 동일)
    GLAM_REDUCTION = 8
    GLAM_SPATIAL_K = 7
    GLAM_ECA_GAMMA = 2
    GLAM_ECA_B     = 1

    # 학습 하이퍼파라미터 (Model D와 완전히 동일)
    BATCH_SIZE    = 128
    NUM_WORKERS   = 4
    WARMUP_EPOCHS = 5
    MAX_EPOCHS    = 100
    PATIENCE      = 10
    BACKBONE_LR   = 3e-5
    HEAD_LR       = 3e-4
    HCF_LR        = 3e-4    # NEW: HCFEncoder는 head와 동일 lr
    WEIGHT_DECAY  = 2e-4
    GRAD_CLIP     = 1.0
    AMP_ENABLED   = os.environ.get('AMP_ENABLED', '1').lower() in {'1', 'true', 'yes', 'on'}
    AMP_DTYPE     = torch.bfloat16
    RUN_INDEX     = int(os.environ.get('RUN_INDEX', 0))
    SEED          = int(os.environ.get('SEED', 42))
    RUN_NAME      = os.environ.get('RUN_NAME', f'run_E_{HCF_VERSION}_seed_{SEED}_{RUN_INDEX + 1:02d}')
    RUN_DIR       = CHECKPOINT_DIR / RUN_NAME

    MEAN          = [0.485, 0.456, 0.406]
    STD           = [0.229, 0.224, 0.225]

CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CFG.RUN_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass

set_seed(CFG.SEED)

print(f"PREPROCESSED_DIR exists: {CFG.PREPROCESSED_DIR.exists()}")
print(f"MANIFESTS_DIR exists:    {CFG.MANIFESTS_DIR.exists()}")
print(f"HCF_DIR exists:          {CFG.HCF_DIR.exists()}")
print(f"CHECKPOINT_DIR:          {CFG.CHECKPOINT_DIR}")
print(f"RUN_DIR:                 {CFG.RUN_DIR}")
print(f"HCF_VERSION:             {CFG.HCF_VERSION}")
print(f"MODEL_NAME:              {CFG.MODEL_NAME}")
print(f"RUN_SEED:                {CFG.SEED}")
print(f"AMP_ENABLED:             {CFG.AMP_ENABLED}")
print(f"AMP_DTYPE:               {CFG.AMP_DTYPE}")


## 3. Manifests 로드

CSV 안의 절대 경로를 본인 환경 경로로 재매핑.


In [ ]:
def remap_path(orig_path, new_root, marker='preprocessed_ssim09_blur10_300'):
    """
    동료의 절대 경로를 본인 PC 경로로 재매핑.
    'preprocessed_ssim09_blur10_300\\' 이후 부분만 살리고 앞을 new_root로 교체.
    
    Args:
        orig_path: 원본 경로 문자열
        new_root: 본인 PC의 preprocessed root 경로
        marker: 경로에서 분기점이 되는 폴더명
    
    Returns:
        본인 PC 경로 (str)
    """
    orig = str(orig_path).replace('/', '\\')   # 슬래시 통일
    
    if marker not in orig:
        return orig   # 매칭 안 되면 원본 반환 (이미 상대 경로일 수도)
    
    # marker 이후의 상대 경로 추출 후 현재 OS 경로로 변환
    relative = orig.split(marker, 1)[1].lstrip('\\').lstrip('/')
    relative_parts = [part for part in relative.replace('\\', '/').split('/') if part]
    
    # 본인 root에 붙이기
    return str(Path(new_root).joinpath(*relative_parts))



def load_split_from_manifest(manifests_dir, preprocessed_root, verify_files=True):
    """Model D와 동일한 split 로직."""
    splits = {}
    for split_name in ['train', 'val', 'test']:
        csv_path = manifests_dir / f'{split_name}.csv'
        if not csv_path.exists():
            raise FileNotFoundError(f'{csv_path} 가 없습니다.')
        df = pd.read_csv(csv_path)
        print(f"\n[{split_name}] CSV 로드: {len(df)} 행")
        df['front_path'] = df['front_path'].apply(lambda p: remap_path(p, preprocessed_root))
        df['back_path']  = df['back_path'].apply(lambda p: remap_path(p, preprocessed_root))
        if verify_files:
            before = len(df)
            df = df[df['front_path'].apply(lambda p: Path(p).exists())].copy()
            df = df[df['back_path'].apply(lambda p: Path(p).exists())].copy()
            after = len(df)
            if before != after:
                print(f"  경로 검증: {before} → {after} ({before-after}개 파일 누락)")
            else:
                print(f"  모든 파일 존재 확인")
        before = len(df)
        df = df[df['condition'].isin([1, 2, 3, 4, 5])].copy()
        if len(df) != before:
            print(f"  condition 필터: {before} → {len(df)}")
        splits[split_name] = df.reset_index(drop=True)
    return splits['train'], splits['val'], splits['test']


# ─── HCF CSV 로드 (Model E 추가) ───
def load_hcf_split(hcf_dir, version, split_name):
    p = hcf_dir / f'hcf_{version}_{split_name}.csv'
    if not p.exists():
        raise FileNotFoundError(
            f'{p} 가 없습니다. extract_hcf_v2.ipynb 를 먼저 실행하세요.'
        )
    return pd.read_csv(p)


def merge_manifest_with_hcf(df_manifest, df_hcf, split_name):
    """item_id로 left join. HCF feature 컬럼들을 manifest에 추가."""
    feature_cols = [c for c in df_hcf.columns if c not in ('item_id', 'condition')]
    merged = df_manifest.merge(
        df_hcf[['item_id'] + feature_cols],
        on='item_id', how='left'
    )
    n_missing = merged[feature_cols].isna().any(axis=1).sum()
    if n_missing > 0:
        print(f"  [{split_name}] HCF 누락 {n_missing} 행 → 0으로 채움")
        merged[feature_cols] = merged[feature_cols].fillna(0)
    else:
        print(f"  [{split_name}] 모든 행 HCF 매칭")
    return merged, feature_cols


# ─── 실행 ───
print("=" * 60)
print("Manifest 로드 (Model D와 동일 split)")
print("=" * 60)
df_train, df_val, df_test = load_split_from_manifest(
    CFG.MANIFESTS_DIR,
    CFG.PREPROCESSED_DIR,
    verify_files=True,
)

print("\n" + "=" * 60)
print(f"HCF 로드 (version={CFG.HCF_VERSION})")
print("=" * 60)
hcf_train = load_hcf_split(CFG.HCF_DIR, CFG.HCF_VERSION, 'train')
hcf_val   = load_hcf_split(CFG.HCF_DIR, CFG.HCF_VERSION, 'val')
hcf_test  = load_hcf_split(CFG.HCF_DIR, CFG.HCF_VERSION, 'test')
print(f"  hcf_train: {hcf_train.shape}")
print(f"  hcf_val:   {hcf_val.shape}")
print(f"  hcf_test:  {hcf_test.shape}")

print("\n" + "=" * 60)
print("Merge by item_id")
print("=" * 60)
df_train, hcf_feature_cols = merge_manifest_with_hcf(df_train, hcf_train, 'train')
df_val,   _                = merge_manifest_with_hcf(df_val,   hcf_val,   'val')
df_test,  _                = merge_manifest_with_hcf(df_test,  hcf_test,  'test')

HCF_DIM = len(hcf_feature_cols)
print(f"\n=== Final ===")
print(f"Train: {len(df_train)}")
print(f"Val:   {len(df_val)}")
print(f"Test:  {len(df_test)}")
print(f"HCF 차원: {HCF_DIM}")
print(f"HCF feature 컬럼 처음 10개: {hcf_feature_cols[:10]}")


In [ ]:
# 분포 확인 (Model D와 동일)
print("=== Train condition 분포 ===")
print(df_train['condition'].value_counts(normalize=True).sort_index().round(3))
print("\n=== Val condition 분포 ===")
print(df_val['condition'].value_counts(normalize=True).sort_index().round(3))
print("\n=== Test condition 분포 ===")
print(df_test['condition'].value_counts(normalize=True).sort_index().round(3))

print("\n=== Train station 분포 ===")
print(df_train['station'].value_counts())

# HCF 통계 (sanity)
print(f"\n=== HCF 통계 (train, 처음 5개 컬럼) ===")
print(df_train[hcf_feature_cols[:5]].describe().T[['mean', 'std', 'min', 'max']].round(3))


In [ ]:
# Sanity check — 첫 행의 파일이 실제로 열리는지 확인
print("=== Sanity Check ===")
sample_row = df_train.iloc[0]
print(f"item_id:    {sample_row['item_id']}")
print(f"condition:  {sample_row['condition']}")
print(f"station:    {sample_row['station']}")
print(f"\nfront_path: {sample_row['front_path']}")
print(f"  exists: {Path(sample_row['front_path']).exists()}")
print(f"back_path:  {sample_row['back_path']}")
print(f"  exists: {Path(sample_row['back_path']).exists()}")

# 실제 이미지 열어보기
try:
    front_img = Image.open(sample_row['front_path'])
    back_img  = Image.open(sample_row['back_path'])
    print(f"\nfront image: size={front_img.size}, mode={front_img.mode}")
    print(f"back image:  size={back_img.size}, mode={back_img.mode}")
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(front_img); axes[0].set_title(f'Front (cond={sample_row["condition"]})'); axes[0].axis('off')
    axes[1].imshow(back_img);  axes[1].set_title(f'Back  (cond={sample_row["condition"]})'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("→ PREPROCESSED_DIR 경로가 맞는지 확인하세요.")

## 4. Dataset

Model D의 GarmentPairDataset에 **HCF tensor** 반환 기능만 추가.
augmentation 등 다른 로직은 Model D와 완전히 동일.


In [ ]:
class GarmentPairDatasetWithHCF(Dataset):
    """Model D의 GarmentPairDataset + HCF tensor 반환.
    
    Augmentation은 Model D와 완전히 동일 (hflip, rotation, ColorJitter).
    유일한 차이: __getitem__이 (front, back, hcf, target) 4-tuple 반환.
    """

    def __init__(self, df, hcf_cols, img_size=300, mode='train'):
        self.df = df.reset_index(drop=True)
        self.hcf_cols = hcf_cols
        self.img_size = img_size
        self.mode = mode
        self.normalize = T.Normalize(mean=CFG.MEAN, std=CFG.STD)
        # 증강 강화: 일부 영역을 가려 과적합 억제 (train 전용)
        self.random_erase = T.RandomErasing(p=0.25, scale=(0.02, 0.12), value=0.0)
        # HCF를 미리 numpy로 cache (학습 중 매번 row.lookup 안 하게)
        self.hcf_matrix = self.df[self.hcf_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        if img.size != (self.img_size, self.img_size):
            img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        return img

    def _paired_train_transform(self, front, back):
        """증강 강화: paired RandomResizedCrop 추가 (front/back 동일 파라미터)."""
        # 동일 crop 영역을 front/back에 똑같이 적용해 view 정합성 유지
        ci, cj, ch, cw = T.RandomResizedCrop.get_params(
            front, scale=(0.75, 1.0), ratio=(0.9, 1.1)
        )
        front = TF.resized_crop(front, ci, cj, ch, cw, [self.img_size, self.img_size])
        back  = TF.resized_crop(back,  ci, cj, ch, cw, [self.img_size, self.img_size])
        if random.random() < 0.5:
            front = TF.hflip(front)
            back  = TF.hflip(back)
        angle = random.uniform(-10, 10)
        front = TF.rotate(front, angle, fill=255)
        back  = TF.rotate(back,  angle, fill=255)
        b = random.uniform(0.8, 1.2)
        c = random.uniform(0.8, 1.2)
        s = random.uniform(0.85, 1.15)
        front = TF.adjust_brightness(front, b)
        back  = TF.adjust_brightness(back,  b)
        front = TF.adjust_contrast(front, c)
        back  = TF.adjust_contrast(back,  c)
        front = TF.adjust_saturation(front, s)
        back  = TF.adjust_saturation(back,  s)
        return front, back

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        front = self._load(row['front_path'])
        back  = self._load(row['back_path'])
        if self.mode == 'train':
            front, back = self._paired_train_transform(front, back)
        front = self.normalize(TF.to_tensor(front))
        back  = self.normalize(TF.to_tensor(back))
        if self.mode == 'train':
            front = self.random_erase(front)
            back  = self.random_erase(back)
        hcf = torch.from_numpy(self.hcf_matrix[idx])
        target = int(row['condition']) - 1
        return front, back, hcf, target


train_ds = GarmentPairDatasetWithHCF(df_train, hcf_feature_cols, CFG.IMG_SIZE, mode='train')
val_ds   = GarmentPairDatasetWithHCF(df_val,   hcf_feature_cols, CFG.IMG_SIZE, mode='val')
test_ds  = GarmentPairDatasetWithHCF(df_test,  hcf_feature_cols, CFG.IMG_SIZE, mode='val')

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

front, back, hcf, target = train_ds[0]
print(f"front: {front.shape}, dtype={front.dtype}")
print(f"back:  {back.shape}, dtype={back.dtype}")
print(f"hcf:   {hcf.shape}, dtype={hcf.dtype}  min={hcf.min():.3f} max={hcf.max():.3f}")
print(f"target (0~4): {target}  → condition = {target+1}")


In [ ]:
# 페어 시각화 + HCF 값 일부 (Model D 시각화 + HCF 표시)
def show_pair_with_hcf(ds, idx, n_hcf_show=8):
    front, back, hcf, target = ds[idx]
    mean = torch.tensor(CFG.MEAN).view(3,1,1)
    std  = torch.tensor(CFG.STD).view(3,1,1)
    front_v = (front * std + mean).clamp(0,1).permute(1,2,0).numpy()
    back_v  = (back  * std + mean).clamp(0,1).permute(1,2,0).numpy()
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(front_v); axes[0].set_title(f'Front (cond={target+1})'); axes[0].axis('off')
    axes[1].imshow(back_v);  axes[1].set_title(f'Back  (cond={target+1})'); axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    print(f'HCF (first {n_hcf_show} / {len(ds.hcf_cols)} cols):')
    for i in range(min(n_hcf_show, len(ds.hcf_cols))):
        print(f'  {ds.hcf_cols[i]:<30s} = {hcf[i].item():.3f}')

for i in range(2):
    show_pair_with_hcf(train_ds, i)


## 5. GLAM (4-way Attention)

| 모듈 | 역할 |
|------|------|
| `LocalChannelAttention` (LCA) | 채널 게이트 (ECA 1D conv, adaptive k) |
| `LocalSpatialAttention` (LSA) | 위치 게이트 (1×1 reduce → 1×1 + dilated 3×3 (d=1,2,3) → concat → 1×1 mask) |
| `GlobalChannelAttention` (GCA) | 채널 self-attention, C×C |
| `GlobalSpatialAttention` (GSA) | 위치 self-attention, HW×HW |

```
local  = x + (LCA(x) + LSA(x))
global = GCA(x) + GSA(x) − x
out    = W_o·x + W_L·local + W_G·global
```


In [ ]:
import math
import torch.nn.functional as F


def adaptive_kernel(channels, gamma=2, b=1):
    k = int(abs((math.log2(channels) + b) / gamma))
    k = k if k % 2 == 1 else k + 1
    return max(k, 3)


class LocalChannelAttention(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        k = adaptive_kernel(channels, gamma, b)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        return x * self.sigmoid(y)


class LocalSpatialAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        c_prime = max(channels // reduction, 1)
        self.reduce = nn.Conv2d(channels, c_prime, kernel_size=1)
        self.branch_1x1 = nn.Conv2d(c_prime, c_prime, kernel_size=1)
        self.branch_d1 = nn.Conv2d(c_prime, c_prime, kernel_size=3, padding=1, dilation=1)
        self.branch_d2 = nn.Conv2d(c_prime, c_prime, kernel_size=3, padding=2, dilation=2)
        self.branch_d3 = nn.Conv2d(c_prime, c_prime, kernel_size=3, padding=3, dilation=3)
        self.out_conv = nn.Conv2d(c_prime * 4, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        F_prime = self.reduce(x)
        b1 = self.branch_1x1(F_prime)
        b2 = self.branch_d1(F_prime)
        b3 = self.branch_d2(F_prime)
        b4 = self.branch_d3(F_prime)
        combined = torch.cat([b1, b2, b3, b4], dim=1)
        mask = self.sigmoid(self.out_conv(combined))
        return x * mask


class GlobalChannelAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        scale = 1.0 / math.sqrt(H * W)

        q = x.view(B, C, -1)
        k = x.view(B, C, -1).transpose(1, 2)
        v = x.view(B, C, -1)

        attn = torch.bmm(q, k) * scale
        attn = attn - attn.max(dim=-1, keepdim=True)[0]
        attn = F.softmax(attn, dim=-1)

        out = torch.bmm(attn, v).view(B, C, H, W)
        return self.gamma * out + x


class GlobalSpatialAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.inter_channels = max(channels // reduction, 1)
        self.q_proj   = nn.Conv2d(channels, self.inter_channels, 1)
        self.k_proj   = nn.Conv2d(channels, self.inter_channels, 1)
        self.v_proj   = nn.Conv2d(channels, self.inter_channels, 1)
        self.out_proj = nn.Conv2d(self.inter_channels, channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.scale = 1.0 / math.sqrt(self.inter_channels)

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.q_proj(x).view(B, self.inter_channels, -1).transpose(1, 2)
        k = self.k_proj(x).view(B, self.inter_channels, -1)
        v = self.v_proj(x).view(B, self.inter_channels, -1).transpose(1, 2)

        attn = torch.bmm(q, k) * self.scale
        attn = attn - attn.max(dim=-1, keepdim=True)[0]
        attn = F.softmax(attn, dim=-1)

        out = torch.bmm(attn, v).transpose(1, 2).contiguous().view(B, self.inter_channels, H, W)
        out = self.out_proj(out)
        return self.gamma * out + x


class GLAM(nn.Module):
    """재설계: Local(채널→공간) + Global(채널→공간) 보정을 학습 게이트로 합성.
    out = x + w_L*(local - x) + w_G*(global - x).
    w 초기 0 → 시작은 identity, x 경로가 하나로 정리돼 스케일이 안정적.
    (기존 _global 의 '- x' 중복가산으로 x가 3중 누적되던 문제 제거)
    """
    def __init__(self, channels, reduction=8, spatial_k=7, eca_gamma=2, eca_b=1):
        super().__init__()
        self.lca = LocalChannelAttention(channels, gamma=eca_gamma, b=eca_b)
        self.lsa = LocalSpatialAttention(channels=channels, reduction=reduction)
        self.gca = GlobalChannelAttention(channels)
        self.gsa = GlobalSpatialAttention(channels, reduction=reduction)

        self.w_L = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.w_G = nn.Parameter(torch.zeros(1, channels, 1, 1))

    def forward(self, x):
        local = self.lsa(self.lca(x))   # 순차 게이팅 (CBAM 순서)
        glob  = self.gsa(self.gca(x))   # 순차 global self-attention (내부 residual 포함)
        return x + self.w_L * (local - x) + self.w_G * (glob - x)


_glam_test = GLAM(
    channels=CFG.FEATURE_DIM,
    reduction=CFG.GLAM_REDUCTION,
    spatial_k=CFG.GLAM_SPATIAL_K,
).to(device)

with torch.no_grad():
    _dummy = torch.randn(2, CFG.FEATURE_DIM, 10, 10).to(device)
    _out = _glam_test(_dummy)
    print(f"GLAM input  shape: {tuple(_dummy.shape)}")
    print(f"GLAM output shape: {tuple(_out.shape)}")
    print(f"LCA adaptive kernel: k = {adaptive_kernel(CFG.FEATURE_DIM)}")
    print(f"LSA c' (reduced channels): {_glam_test.lsa.reduce.out_channels}")
    print(f"LSA branches: 1x1 + dilated 3x3 (d=1,2,3) -> 4 streams")
    print(f"GCA scale: 1/sqrt(100) = {1/math.sqrt(100):.4f}")
    print(f"GSA scale: 1/sqrt({_glam_test.gsa.inter_channels}) = {_glam_test.gsa.scale:.4f}")

print(f"GLAM params total: {sum(p.numel() for p in _glam_test.parameters()):,}")
print(f"  LCA: {sum(p.numel() for p in _glam_test.lca.parameters()):,}")
print(f"  LSA: {sum(p.numel() for p in _glam_test.lsa.parameters()):,}")
print(f"  GCA: {sum(p.numel() for p in _glam_test.gca.parameters()):,}")
print(f"  GSA: {sum(p.numel() for p in _glam_test.gsa.parameters()):,}")

del _glam_test, _dummy, _out
torch.cuda.empty_cache()


## 6. Model — ModelD = ModelC + HCFEncoder

Model C 구조 그대로. 추가된 것:
- `HCFEncoder`: HCF(N) → BN → MLP → embed(64)
- Head 입력: 3072 → 3136 (HCF embed concat)
- `forward(front, back, hcf)` 3-input

Backbone/GLAM/GAP/head 구조 자체는 Model D와 동일.
`freeze_backbone()`도 Model D와 동일 (HCFEncoder는 영향 받지 않음).


In [ ]:
class HCFEncoder(nn.Module):
    """HCF (N차원) → embed (HCF_EMBED_DIM) 변환 MLP.
    BN으로 입력 스케일 안정화. GELU + Dropout.
    """
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class ModelE(nn.Module):
    """Model D + HCF 분기. backbone/GLAM/GAP/head 구조는 Model D와 동일."""

    def __init__(self, backbone_name='efficientnet_b3', pretrained=True,
                 feature_dim=1536, num_classes=5, dropout=0.3,
                 glam_reduction=8, glam_spatial_k=7,
                 glam_eca_gamma=2, glam_eca_b=1,
                 hcf_dim=148, hcf_hidden=128, hcf_embed=64, hcf_dropout=0.2):
        super().__init__()
        # Backbone — Model D와 동일하지만 timm 자동 다운로드 방식
        # (Model D는 로컬 .safetensors 경로 명시. 동일 가중치 efficientnet_b3.ra2_in1k)
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
            global_pool='',
        )
        if pretrained:
            if not CFG.PRETRAINED_WEIGHTS.exists():
                raise FileNotFoundError(f'Pretrained weights not found: {CFG.PRETRAINED_WEIGHTS}')
            from safetensors.torch import load_file
            state_dict = load_file(str(CFG.PRETRAINED_WEIGHTS), device='cpu')
            self.backbone.load_state_dict(state_dict, strict=False)

        # GLAM — Model D와 완전히 동일
        self.glam = GLAM(
            channels=feature_dim,
            reduction=glam_reduction,
            spatial_k=glam_spatial_k,
            eca_gamma=glam_eca_gamma,
            eca_b=glam_eca_b,
        )

        self.gap = nn.AdaptiveAvgPool2d(1)

        # NEW: HCF Encoder
        self.hcf_encoder = HCFEncoder(
            in_dim=hcf_dim,
            hidden_dim=hcf_hidden,
            out_dim=hcf_embed,
            dropout=hcf_dropout,
        )

        # 재설계: front/back/|diff| 결합 → proj, 그 위에 HCF FiLM 변조
        proj_dim = feature_dim // 2
        # 결합: [f_a, f_b, |f_a-f_b|, f_a*f_b] = 4*feature_dim
        # (곱항은 스케일이 큼 → 앞단 LayerNorm 이 정규화해 안정화)
        self.img_proj = nn.Sequential(
            nn.LayerNorm(feature_dim * 4),
            nn.Linear(feature_dim * 4, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        # FiLM: HCF embed → (gamma, beta) 로 이미지 특징을 변조. 0-init → 시작은 identity
        self.film = nn.Linear(hcf_embed, proj_dim * 2)
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)

        head_in = proj_dim + hcf_embed
        self.head = nn.Sequential(
            nn.LayerNorm(head_in),
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def extract_features(self, x):
        """Model D의 extract_features와 완전히 동일."""
        feat = self.backbone(x)
        feat = self.glam(feat)
        feat = self.gap(feat)
        feat = feat.flatten(1)
        return feat

    def forward(self, front, back, hcf):
        f_a = self.extract_features(front)
        f_b = self.extract_features(back)
        # front/back 대칭 결합: 두 뷰 + 차이(|diff|) + 상호작용(곱항)
        f_img = torch.cat([
            f_a, f_b, torch.abs(f_a - f_b), f_a * f_b
        ], dim=1)                                                   # (B, 4*feature_dim)
        f_img = self.img_proj(f_img)                                # (B, proj_dim)
        f_hcf = self.hcf_encoder(hcf)                               # (B, hcf_embed)
        # FiLM 변조: HCF가 이미지 특징의 scale/shift 를 조절
        gamma, beta = self.film(f_hcf).chunk(2, dim=1)
        f_mod = f_img * (1 + gamma) + beta
        f_all = torch.cat([f_mod, f_hcf], dim=1)                    # (B, proj_dim + hcf_embed)
        logits = self.head(f_all)
        return logits

    def freeze_backbone(self):
        """Model D와 동일. HCFEncoder/GLAM/head는 영향 없음."""
        for p in self.backbone.parameters():
            p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = True


model = ModelE(
    backbone_name=CFG.BACKBONE_NAME,
    pretrained=CFG.PRETRAINED,
    feature_dim=CFG.FEATURE_DIM,
    num_classes=CFG.NUM_CLASSES,
    dropout=CFG.DROPOUT,
    glam_reduction=CFG.GLAM_REDUCTION,
    glam_spatial_k=CFG.GLAM_SPATIAL_K,
    glam_eca_gamma=CFG.GLAM_ECA_GAMMA,
    glam_eca_b=CFG.GLAM_ECA_B,
    hcf_dim=HCF_DIM,
    hcf_hidden=CFG.HCF_HIDDEN_DIM,
    hcf_embed=CFG.HCF_EMBED_DIM,
    hcf_dropout=CFG.HCF_DROPOUT,
).to(device)

model.eval()
with torch.no_grad():
    dummy_f   = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(device)
    dummy_b   = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(device)
    dummy_hcf = torch.randn(2, HCF_DIM).to(device)
    out = model(dummy_f, dummy_b, dummy_hcf)
model.train()
    print(f"Output shape: {out.shape}")

backbone_params = sum(p.numel() for p in model.backbone.parameters())
glam_params     = sum(p.numel() for p in model.glam.parameters())
hcf_params      = sum(p.numel() for p in model.hcf_encoder.parameters())
head_params     = sum(p.numel() for p in model.head.parameters())
total           = sum(p.numel() for p in model.parameters())

print(f"Backbone params:    {backbone_params:,}")
print(f"GLAM params:        {glam_params:,}")
print(f"HCF Encoder params: {hcf_params:,}  (input dim = {HCF_DIM})")
print(f"Head params:        {head_params:,}  (input dim = {CFG.FEATURE_DIM*2 + CFG.HCF_EMBED_DIM})")
print(f"Total params:       {total:,}")
print(f"\n[재설계] front/back/|diff| 결합 → img_proj, HCF는 FiLM 변조 + concat")
proj_params = sum(p.numel() for p in model.img_proj.parameters())
film_params = sum(p.numel() for p in model.film.parameters())
print(f"img_proj params:    {proj_params:,}")
print(f"FiLM params:        {film_params:,}")


## 7. Loss 및 평가 지표

Model D와 완전히 동일.
CrossEntropyLoss + sklearn 분류 지표 (acc, balanced_acc, f1, kappa).
kappa는 unweighted (Model D와 일치). condition은 ordinal이지만 D와 paired 비교 위해 동일 metric 사용.


In [ ]:
def predict(logits):
    return logits.argmax(dim=1)


def compute_metrics(preds, targets):
    preds   = np.asarray(preds,   dtype=np.int64)
    targets = np.asarray(targets, dtype=np.int64)
    return {
        'acc':          float(accuracy_score(targets, preds)),
        'balanced_acc': float(balanced_accuracy_score(targets, preds)),
        'f1_weighted':  float(f1_score(targets, preds, average='weighted', zero_division=0)),
        'f1_macro':     float(f1_score(targets, preds, average='macro',    zero_division=0)),
        'kappa':        float(cohen_kappa_score(targets, preds)),
    }

_class_counts = df_train['condition'].astype(int).value_counts().sort_index()
_counts = np.array([_class_counts.get(c, 0) for c in range(1, CFG.NUM_CLASSES + 1)], dtype=np.float64)
_class_weights = _counts.sum() / (CFG.NUM_CLASSES * np.clip(_counts, 1, None))
_class_weights = torch.tensor(_class_weights, dtype=torch.float32, device=device)
print(f"class counts:  {_counts.astype(int).tolist()}")
print(f"class weights: {_class_weights.cpu().numpy().round(3).tolist()}")

criterion = nn.CrossEntropyLoss(weight=_class_weights, label_smoothing=0.05)

# smoke test
fake_logits = torch.randn(4, 5)
fake_target = torch.tensor([0, 2, 4, 1])
loss = criterion(fake_logits.to(device), fake_target.to(device))
preds_smoke = predict(fake_logits)
print(f"loss smoke: {loss.item():.4f}")
print(f"pred smoke: {preds_smoke.tolist()}")
print(f"metric smoke: {compute_metrics(preds_smoke, fake_target)}")


## 8. 학습 / 검증 함수

Model D의 train_one_epoch / evaluate와 거의 동일. 차이는 단 하나: batch unpacking이
(front, back, target) → (front, back, hcf, target) 4-tuple이고, model forward에 hcf도 전달.


In [ ]:
def train_one_epoch(model, loader, optimizer, epoch_idx):
    model.train()
    total_loss = 0.0
    n_samples = 0
    all_preds, all_targets = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch_idx} [train]", leave=False)
    for front, back, hcf, target in pbar:           # Model D 대비: hcf 추가
        front  = front.to(device, non_blocking=True)
        back   = back.to(device, non_blocking=True)
        hcf    = hcf.to(device, non_blocking=True)  # NEW
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=CFG.AMP_ENABLED, dtype=CFG.AMP_DTYPE):
            logits = model(front, back, hcf)        # NEW: hcf 전달
            loss = criterion(logits, target)

        if not torch.isfinite(loss):
            print("Non-finite train loss detected")
            print("target min/max:", target.min().item(), target.max().item())
            print("logits min/max:", logits.detach().min().item(), logits.detach().max().item())
            print("hcf min/max/has_nan:", hcf.min().item(), hcf.max().item(), torch.isnan(hcf).any().item())
            raise RuntimeError("NaN/Inf train loss")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.GRAD_CLIP)
        optimizer.step()

        with torch.no_grad():
            preds = predict(logits)
            all_preds.append(preds.cpu().numpy())
            all_targets.append(target.cpu().numpy())

        total_loss += loss.item() * front.size(0)
        n_samples += target.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / n_samples
    metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_targets))
    metrics['loss'] = avg_loss
    return metrics


@torch.no_grad()
def evaluate(model, loader, desc='val'):
    model.eval()
    total_loss = 0.0
    n_samples = 0
    all_preds, all_targets = [], []

    for front, back, hcf, target in tqdm(loader, desc=f"[{desc}]", leave=False):
        front  = front.to(device, non_blocking=True)
        back   = back.to(device, non_blocking=True)
        hcf    = hcf.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        with autocast(enabled=CFG.AMP_ENABLED, dtype=CFG.AMP_DTYPE):
            logits = model(front, back, hcf)
            loss = criterion(logits, target)

        if not torch.isfinite(loss):
            print("Non-finite validation loss detected")
            print("target min/max:", target.min().item(), target.max().item())
            print("logits min/max:", logits.detach().min().item(), logits.detach().max().item())
            raise RuntimeError("NaN/Inf validation loss")

        preds = predict(logits)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(target.cpu().numpy())
        total_loss += loss.item() * front.size(0)
        n_samples += target.size(0)

    avg_loss = total_loss / n_samples
    metrics = compute_metrics(np.concatenate(all_preds), np.concatenate(all_targets))
    metrics['loss'] = avg_loss
    return metrics, np.concatenate(all_preds), np.concatenate(all_targets)


## 9. 메인 학습 루프

Model D와 완전히 동일한 학습 일정:
- Stage 1 (3 epoch): backbone freeze → **GLAM + HCFEncoder + head** 학습
  (Model D는 GLAM + head만 학습. HCFEncoder가 추가되어 학습 대상)
- Stage 2 (~97 epoch): backbone unfreeze, lr 차등 (backbone 3e-5, 나머지 3e-4)
- patience=5 early stop


In [ ]:
def seed_worker(worker_id):
    """Model D와 동일."""
    worker_seed = CFG.SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

loader_generator = torch.Generator()
loader_generator.manual_seed(CFG.SEED)

train_loader = DataLoader(
    train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
    num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=(CFG.NUM_WORKERS > 0),
    worker_init_fn=seed_worker, generator=loader_generator,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True,
    persistent_workers=(CFG.NUM_WORKERS > 0),
    worker_init_fn=seed_worker,
)
test_loader = DataLoader(
    test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=True,
    persistent_workers=(CFG.NUM_WORKERS > 0),
    worker_init_fn=seed_worker,
)


# Optimizer: Model D는 3 param groups (backbone/glam/head).
#            Model E는 4 param groups (backbone/glam/hcf/head). HCFEncoder는 head와 동일 lr.
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(),    'lr': CFG.BACKBONE_LR},
    {'params': model.glam.parameters(),        'lr': CFG.HEAD_LR},
    {'params': model.hcf_encoder.parameters(), 'lr': CFG.HCF_LR},     # NEW
    # img_proj + film + head 를 head 그룹에 함께 포함 (이전엔 img_proj/film 누락 → 학습 안 됨)
    {'params': (list(model.img_proj.parameters())
                + list(model.film.parameters())
                + list(model.head.parameters())), 'lr': CFG.HEAD_LR},
], weight_decay=CFG.WEIGHT_DECAY)

# LR이 거의 안 떨어지던 문제 수정: val_loss plateau에 반응해 lr을 절반씩 감소
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, min_lr=1e-6,
)


print(f"학습 준비 완료 (device={device})")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Param groups (Model D 대비 hcf_encoder 추가):")
print(f"  backbone    (lr={CFG.BACKBONE_LR})")
print(f"  glam        (lr={CFG.HEAD_LR})")
print(f"  hcf_encoder (lr={CFG.HCF_LR})  ← NEW")
print(f"  head        (lr={CFG.HEAD_LR})")


In [ ]:
history = {'train': [], 'val': []}
best_val_loss = float('inf')
patience_counter = 0
best_epoch = -1
best_path = CFG.RUN_DIR / f'best_{CFG.MODEL_NAME}.pt'

def _save_best(epoch, val_m):
    global best_val_loss, best_epoch, patience_counter
    if np.isfinite(val_m['loss']) and val_m['loss'] < best_val_loss:
        best_val_loss, best_epoch, patience_counter = val_m['loss'], epoch, 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(), 'val_metrics': val_m,
                    'config': {k: str(v) for k, v in vars(CFG).items() if not k.startswith('_')},
                    'hcf_dim': HCF_DIM, 'hcf_version': CFG.HCF_VERSION},
                   best_path)
        print(f'  Best updated (val_loss {best_val_loss:.4f})')
    else:
        patience_counter += 1
        if not np.isfinite(val_m['loss']):
            print('  Skipped checkpoint: non-finite validation loss')

print('=' * 60)
print('Stage 1: Warmup - backbone freeze (GLAM + HCFEncoder + head 학습)')
print('=' * 60)
model.freeze_backbone()
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  trainable params: {n_trainable:,}')

for epoch in range(CFG.WARMUP_EPOCHS):
    train_m = train_one_epoch(model, train_loader, optimizer, epoch)
    val_m, _, _ = evaluate(model, val_loader, desc='val')
    history['train'].append({'epoch': epoch, **train_m})
    history['val'].append({'epoch': epoch, **val_m})
    print(f'[Warmup {epoch+1}/{CFG.WARMUP_EPOCHS}] '
          f'train_loss={train_m["loss"]:.4f} acc={train_m["acc"]:.3f} | '
          f'val_loss={val_m["loss"]:.4f} acc={val_m["acc"]:.3f} f1={val_m["f1_weighted"]:.3f}')
    _save_best(epoch, val_m)

print('\n' + '=' * 60)
print('Stage 2: Full fine-tuning - backbone unfreeze')
print('=' * 60)
model.unfreeze_backbone()
patience_counter = 0
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  trainable params: {n_trainable:,}')

for epoch in range(CFG.WARMUP_EPOCHS, CFG.MAX_EPOCHS):
    train_m = train_one_epoch(model, train_loader, optimizer, epoch)
    val_m, _, _ = evaluate(model, val_loader, desc='val')
    scheduler.step(val_m['loss'])
    history['train'].append({'epoch': epoch, **train_m})
    history['val'].append({'epoch': epoch, **val_m})
    cur_lr_b = optimizer.param_groups[0]['lr']
    cur_lr_g = optimizer.param_groups[1]['lr']
    cur_lr_x = optimizer.param_groups[2]['lr']
    cur_lr_h = optimizer.param_groups[3]['lr']
    print(f'[Epoch {epoch+1}/{CFG.MAX_EPOCHS}] '
          f'train_loss={train_m["loss"]:.4f} acc={train_m["acc"]:.3f} | '
          f'val_loss={val_m["loss"]:.4f} acc={val_m["acc"]:.3f} '
          f'f1={val_m["f1_weighted"]:.3f} kappa={val_m["kappa"]:.3f} | '
          f'lr_b={cur_lr_b:.2e} lr_g={cur_lr_g:.2e} lr_x={cur_lr_x:.2e} lr_h={cur_lr_h:.2e}')
    _save_best(epoch, val_m)
    if patience_counter >= CFG.PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1} (no improvement for {CFG.PATIENCE} epochs)')
        break

print(f'\n학습 완료. Best epoch: {best_epoch+1}, Best val_loss: {best_val_loss:.4f}')


## 10. 학습 곡선

In [ ]:
hist_train = pd.DataFrame(history['train'])
hist_val   = pd.DataFrame(history['val'])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(hist_train['epoch'], hist_train['loss'], label='train')
axes[0,0].plot(hist_val['epoch'],   hist_val['loss'],   label='val')
axes[0,0].set_title('Loss'); axes[0,0].set_xlabel('epoch'); axes[0,0].legend()

axes[0,1].plot(hist_train['epoch'], hist_train['acc'], label='train')
axes[0,1].plot(hist_val['epoch'],   hist_val['acc'],   label='val')
axes[0,1].set_title('Accuracy'); axes[0,1].set_xlabel('epoch'); axes[0,1].legend()

axes[1,0].plot(hist_train['epoch'], hist_train['f1_weighted'], label='train')
axes[1,0].plot(hist_val['epoch'],   hist_val['f1_weighted'],   label='val')
axes[1,0].set_title('F1 (weighted)'); axes[1,0].set_xlabel('epoch'); axes[1,0].legend()

axes[1,1].plot(hist_train['epoch'], hist_train['kappa'], label='train')
axes[1,1].plot(hist_val['epoch'],   hist_val['kappa'],   label='val')
axes[1,1].set_title("Cohen's Kappa"); axes[1,1].set_xlabel('epoch'); axes[1,1].legend()

plt.tight_layout()
plt.show()


## 11. Test 평가

In [ ]:
ckpt = torch.load(CFG.RUN_DIR / f'best_{CFG.MODEL_NAME}.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f"Loaded best checkpoint from epoch {ckpt['epoch']+1}")
print(f"Saved val metrics: {ckpt['val_metrics']}")

test_metrics, test_preds, test_targets = evaluate(model, test_loader, desc='test')

print("\n=== Test Metrics ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

pd.DataFrame(history['train']).to_csv(CFG.RUN_DIR / f'{CFG.MODEL_NAME}_history_train.csv', index=False)
pd.DataFrame(history['val']).to_csv(CFG.RUN_DIR / f'{CFG.MODEL_NAME}_history_val.csv', index=False)
cm = confusion_matrix(test_targets, test_preds, labels=[0, 1, 2, 3, 4])
pd.DataFrame(cm, index=[1,2,3,4,5], columns=[1,2,3,4,5]).to_csv(CFG.RUN_DIR / f'{CFG.MODEL_NAME}_confusion.csv')


In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_targets, test_preds, labels=[0,1,2,3,4])
class_names = ['1', '2', '3', '4', '5']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted condition')
plt.ylabel('True condition')
plt.title(f'Test Confusion Matrix (Model D)\n'
          f'Acc={test_metrics["acc"]:.3f}, F1w={test_metrics["f1_weighted"]:.3f}, '
          f'Kappa={test_metrics["kappa"]:.3f}')
plt.tight_layout()
plt.show()


In [ ]:
result = {
    'model': CFG.MODEL_NAME,
    'backbone': CFG.BACKBONE_NAME,
    'run_name': CFG.RUN_NAME,
    'seed': int(CFG.SEED),
    'hcf_version': CFG.HCF_VERSION,                       # NEW (D 대비)
    'hcf_dim': HCF_DIM,                                   # NEW
    'hcf_embed_dim': CFG.HCF_EMBED_DIM,                   # NEW
    'config': {k: str(v) for k, v in vars(CFG).items() if not k.startswith('_')},
    'best_epoch': int(ckpt['epoch']),
    'val_metrics': {k: float(v) for k, v in ckpt['val_metrics'].items()},
    'test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'data': {
        'n_train': len(df_train),
        'n_val':   len(df_val),
        'n_test':  len(df_test),
    },
    'env': {
        'torch': torch.__version__,
        'timm': timm.__version__,
        'gpu': torch.cuda.get_device_name(0),
    },
    # NEW: Model D와 paired 비교 위한 메타정보
    'paired_with_model_d': {
        'identical': [
            'split (same manifest)', 'augmentation', 'optimizer (AdamW)',
            'lr (3e-5 backbone, 3e-4 others)', 'schedule (CosineAnnealing)',
            'batch_size=128', 'max_epochs=100', 'patience=5', 'seed=42',
            'AMP bfloat16', 'GLAM hyperparameters',
            'kappa metric (unweighted, sklearn default)',
        ],
        'changed': [
            f'+ HCF Encoder (in={HCF_DIM}, hidden={CFG.HCF_HIDDEN_DIM}, out={CFG.HCF_EMBED_DIM})',
            f'+ Head input dim {CFG.FEATURE_DIM*2} → {CFG.FEATURE_DIM*2 + CFG.HCF_EMBED_DIM}',
            f'+ Optimizer param groups 3 → 4 (HCFEncoder, lr={CFG.HCF_LR})',
        ],
    },
}

with open(CFG.RUN_DIR / f'{CFG.MODEL_NAME}_result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("결과 저장 완료:", CFG.RUN_DIR / f'{CFG.MODEL_NAME}_result.json')
print(json.dumps(result, indent=2, ensure_ascii=False))
